# Evaluate Cosmos3-Nano on Perception Test validation MCQ

**Public download → validate coverage → native inference → local top1.** Default smoke selects five questions (IDs 0–4) from `video_8241`, stable indices 13404–13408. Full validation is explicit opt-in: 19,140 questions / 5,260 MCQ videos, **not the held-out test split**.

Dataset attribution: [Perception Test authors / Google DeepMind](https://github.com/google-deepmind/perception_test), [paper](https://arxiv.org/abs/2305.13786). Their code is Apache 2.0 and other materials [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); retain attribution/license link and identify adaptations. This launchable is an adaptation, not author endorsement or a customer-ingestion evaluation agent.

The public Nano checkpoint is pinned to `e59a53c25979a090fa8706c9acc0c254a6e89b92`. This committed notebook is unexecuted; no empirical validation or reference-score parity is asserted.

## 1 · Choose scope and execution mode

The external `PERCEPTION_TEST_DATA_DIR` is mounted at `/perception-test`; cache and results are separate. No model/data credentials, Docker socket or dataset artifacts are embedded. `RUN_FULL = True` selects full validation directly without rerunning smoke.

For **score-only**, choose explicit `SCORE_SCOPE` and exactly one of `PREDICTIONS_FILE` or `EXISTING_WORK`. Only pinned metadata is needed: no video, model, GPU or endpoint. Set `DOWNLOAD_PUBLIC = False` for already-staged offline scoring.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone
from IPython.display import FileLink, display

SCORE_ONLY = os.environ.get('EVAL_SCORE_ONLY') == '1'
PREDICTIONS_FILE = None
EXISTING_WORK = None
SCORE_SCOPE = 'smoke'  # Choose 'full' explicitly for 19,140 existing predictions.
RUN_FULL = False
DOWNLOAD_PUBLIC = True
NFRAME = 16  # Fixed launchable prototype protocol.
API_NPROC = 1
DATA = Path(os.environ.get('PERCEPTION_TEST_DATA_DIR', '/perception-test'))
SCOPE = SCORE_SCOPE if SCORE_ONLY else ('full' if RUN_FULL else 'smoke')
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/results') / f'perception-test-{SCOPE}-{STAMP}'
SCORE_DIR = Path('/results') / f'perception-test-score-{SCOPE}-{STAMP}'
CLI = [sys.executable, '-u', '/opt/vlmevalkit/launchable/launchable.py']

def invoke(command):
    # Stream each line into Jupyter, not a buffer held until the run ends.
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

if SCOPE not in {'smoke', 'full'}:
    raise ValueError('Choose explicit smoke or full scope.')
if SCORE_ONLY and (PREDICTIONS_FILE is None) == (EXISTING_WORK is None):
    raise ValueError('Set exactly one of PREDICTIONS_FILE or EXISTING_WORK.')
print(('Score existing predictions' if SCORE_ONLY else 'Infer and score'), SCOPE)
print('Data:', DATA, '| frames:', NFRAME, '| API workers:', API_NPROC)

## 2 · Download and prepare the selected public scope

The helper uses direct official public URLs, pinned SHA-256, and safe extraction. Source repository revision: `3938d2f1ba3a6b502025741cea4cd73c7b3bdfaf`. Metadata comprises `mc_question_valid.json`, `cut_frame_mapping_valid.json`, and the attribution README. Smoke downloads the approximately 225 MB official sample archive; full downloads the 75,375,143,328-byte official validation archive, plus extraction space.

Inference preparation requires every selected video before contacting the model. Score-only downloads metadata only and prepares with `require_media=False`. Canonical annotation/cut/scope fingerprints are independent of video presence, with media hashes recorded separately.

In [ ]:
if DOWNLOAD_PUBLIC:
    invoke(CLI + ['download', '--data-root', str(DATA),
                  '--scope', 'metadata' if SCORE_ONLY else SCOPE])
else:
    print('Using staged pinned inputs; no download.')
scope_flag = ['--smoke'] if SCOPE == 'smoke' else []
invoke(CLI + ['prepare', '--data-root', str(DATA)] + scope_flag
       + (['--metadata-only'] if SCORE_ONLY else []))

## 3 · Infer and score locally, or rescore existing predictions

**Prototype protocol:** exactly 16 uniform frames within official source cutpoints; `-1` means all frames, positive cuts are exclusive stop frames. Bad cuts fail rather than clamp. Aspect-preserving downscale only to maximum side 768, no crop. Sampled images go directly to the endpoint, never re-encoded video. This is not a reference-score or official letter-parser parity claim.

Native `CosmosReason2` generation uses temperature 0, seed 1, no thinking, max 8,192 output tokens. BYOM must accept sampled-image messages; configure endpoint/model/revision before host setup and keep API keys environment-only.

Progress streams here and into `run.log`. Fresh output directories prevent stale reuse. Exact selected IDs are required. Blank, missing, duplicate, unexpected or transport-failure rows reject incomplete runs before scoring. Nonempty malformed/ambiguous choices count wrong and invalid, not dropped; literal text `nan` remains model text. Trusted labels come only from pinned annotations, never prediction-supplied answers. No API judge, dummy answer or partial-full-score fallback.

In [ ]:
if SCORE_ONLY:
    source = (['--work-dir', str(EXISTING_WORK)] if EXISTING_WORK is not None
              else ['--predictions', str(PREDICTIONS_FILE)])
    invoke(CLI + ['score', '--data-root', str(DATA), '--scope', SCOPE,
                  '--score-dir', str(SCORE_DIR), '--nframe', str(NFRAME)] + source)
    SUMMARY = SCORE_DIR / 'summary.json'
else:
    # Controller validates coverage before checking the model endpoint.
    invoke(CLI + ['run', '--data-root', str(DATA), '--mode', 'all',
                  '--nframe', str(NFRAME), '--api-nproc', str(API_NPROC),
                  '--work-dir', str(WORK)] + scope_flag)
    SUMMARY = WORK / 'scores' / 'summary.json'

## 4 · Human result and durable artifacts

Top1 is correct option ID / **all selected questions**, with per-area, per-reasoning and per-tag diagnostics in the summary. Smoke is not a full benchmark. Artifact provenance records annotation/cut identities, separate media hashes, prediction hash, source/model revisions and frame protocol. Controller artifacts include useful UTC start/end and measured elapsed time, not invented cost or peak memory.

In [ ]:
summary = json.loads(SUMMARY.read_text())
metric = summary['primary_metric']
print(f"Perception Test validation — {SCOPE}: {summary['status']}")
print(f"Top1: {metric['value']:.2%} ({metric['correct']}/{metric['total']})")
print('Coverage:', json.dumps(summary['coverage']))
print('Prototype protocol; no reference-score parity claim.')
artifacts = [SUMMARY, SUMMARY.parent / 'controller.json']
if not SCORE_ONLY:
    artifacts += [WORK / 'run.json', WORK / 'run.log', WORK / 'config.json']
for artifact in artifacts:
    if artifact.is_file():
        display(FileLink(str(artifact), url_prefix='/files'))

## Stop and retain results

In the **host terminal**, `docker compose --profile local stop` preserves mounted state. `bash setup.sh --score-only` starts only authenticated Jupyter. Evaluator service: `perception-eval`; Compose project: `cosmos3-perception-test`. There is no Docker socket or host model port. Stopping containers does not stop Brev VM billing; stop the instance separately.

Do not commit credentials, videos, labels, predictions or answer-bearing artifacts. Clear outputs before sharing. The supported deployment runtime is native Linux x86-64. A source-only change does not demonstrate notebook/GPU execution or full validation reproduction.